In [ ]:
import psycopg as pg
import pandas as pd
import os
import mlflow
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from catboost import CatBoostClassifier
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

In [4]:
TABLE_NAME = "users_churn" # таблица с данными в postgres 

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

ASSETS_DIR = "assets"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

os.makedirs(ASSETS_DIR, exist_ok=True)

pd.options.display.max_columns = 100
pd.options.display.max_rows = 64

sns.set_style("white")
sns.set_theme(style="whitegrid") 

In [19]:
# определяем основные credentials, которые нужны для подключения к MLflow
# важно, что credentials мы передаём для себя как пользователей Tracking Service
# у вас должен быть доступ к бакету, в который вы будете складывать артефакты
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net" #endpoint бакета от YandexCloud
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID") # получаем id ключа бакета, к которому подключён MLFlow, из .env
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY") # получаем ключ бакета, к которому подключён MLFlow, из .env
# определяем глобальные переменные
# поднимаем MLflow локально
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

EXPERIMENT_NAME = "churn_vds"
RUN_NAME = "eda"
REGISTRY_MODEL_NAME = "column_transformer"

In [43]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

# определим название таблицы, в которой хранятся наши данные.
TABLE_NAME = "users_churn"

# эта конструкция создаёт контекстное управление для соединения с базой данных 
# оператор with гарантирует, что соединение будет корректно закрыто после выполнения всех операций 
# закрыто оно будет даже в случае ошибки, чтобы не допустить "утечку памяти"
with pg.connect(**connection) as conn:

# создаёт объект курсора для выполнения запросов к базе данных
# с помощью метода execute() выполняется SQL-запрос для выборки данных из таблицы TABLE_NAME
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns).dropna(subset=num_columns)

In [44]:
# определение категориальных колонок, которые будут преобразованы
cat_columns = ["type", "payment_method", "internet_service", "gender"]

# создание объекта OneHotEncoder для преобразования категориальных переменных
# auto - автоматическое определение категорий
# ignore - игнорировать ошибки, если встречается неизвестная категория
# max_categories - максимальное количество уникальных категорий
# sparse_output - вывод в виде разреженной матрицы, если False, то в виде обычного массива
# drop="first" - удаляет первую категорию, чтобы избежать ловушки мультиколлинеарности
encoder_oh = OneHotEncoder(categories='auto', handle_unknown='ignore', max_categories=10, sparse_output=False, drop='first')

# применение OneHotEncoder к данным. Преобразование категориальных данных в массив
encoded_features = encoder_oh.fit_transform(df[cat_columns].to_numpy())

# преобразование полученных признаков в DataFrame и установка названий колонок
# get_feature_names_out() - получение имён признаков после преобразования
encoded_df = pd.DataFrame(encoded_features, columns=encoder_oh.get_feature_names_out())

# конкатенация исходного DataFrame с новым DataFrame, содержащим закодированные категориальные признаки
# axis=1 означает конкатенацию по колонкам
obj_df = pd.concat([df, encoded_df], axis=1)

In [45]:
num_columns = ["monthly_charges", "total_charges"]
num_df = df[num_columns]

n_knots = 3
degree_spline = 4
n_quantiles=100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None


# SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)
encoded_features = encoder_spl.fit_transform(df[num_columns].to_numpy())

# encoded_df = pd.DataFrame(
#     encoded_features, 
#     columns=encoder_spl.get_feature_names_out(num_columns)
# )
# num_df = pd.concat([num_df, encoded_df], axis=1)


# QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)
encoded_features = encoder_q.fit_transform(df[num_columns].to_numpy())

# encoded_df = pd.DataFrame(
#     encoded_features, 
#     columns=encoder_q.get_feature_names_out(num_columns)
# )
# encoded_df.columns = [col + f"_q_{n_quantiles}" for col in num_columns]
# num_df = pd.concat([num_df, encoded_df], axis=1)


# RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(df[num_columns].to_numpy())

# encoded_df = pd.DataFrame(
#     encoded_features, 
#     columns=encoder_rb.get_feature_names_out(num_columns)
# )
# encoded_df.columns = [col + f"_robust" for col in num_columns]
# num_df = pd.concat([num_df, encoded_df], axis=1)


# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(df[num_columns].to_numpy())

# encoded_df = pd.DataFrame(
#     encoded_features, 
#     columns=encoder_pol.get_feature_names_out(num_columns)
# )
# encoded_df = encoded_df.iloc[:, 1 + len(num_columns):]
# encoded_df.columns = [col + f"_poly" for col in num_columns]
# num_df = pd.concat([num_df, encoded_df], axis=1)

# KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(n_bins=n_bins, encode=encode, strategy=strategy, subsample=subsample)
encoded_features = encoder_kbd.fit_transform(df[num_columns].to_numpy())

# encoded_df = pd.DataFrame(
#     encoded_features, 
#     columns=encoder_kbd.get_feature_names_out(num_columns)
# )
# encoded_df.columns = [col + f"_bin" for col in num_columns]
# num_df = pd.concat([num_df, encoded_df], axis=1)


In [46]:
numeric_transformer = ColumnTransformer(
    transformers=[
        ('spl', encoder_spl, num_columns), 
        ('q', encoder_q, num_columns), 
        ('rb', encoder_rb, num_columns), 
        ('pol', encoder_pol, num_columns), 
        ('kbd', encoder_kbd, num_columns)
        ]
)

categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_columns), ('cat', categorical_transformer, cat_columns)
        ], n_jobs=-1
)

encoded_features = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(
    encoded_features, 
    columns=preprocessor.get_feature_names_out()
)

df = pd.concat([df, transformed_df.set_index(df.index)], axis=1)
df.head()

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target,num__spl__monthly_charges_sp_0,num__spl__monthly_charges_sp_1,num__spl__monthly_charges_sp_2,num__spl__monthly_charges_sp_3,num__spl__monthly_charges_sp_4,num__spl__monthly_charges_sp_5,num__spl__total_charges_sp_0,num__spl__total_charges_sp_1,num__spl__total_charges_sp_2,num__spl__total_charges_sp_3,num__spl__total_charges_sp_4,num__spl__total_charges_sp_5,num__q__monthly_charges,num__q__total_charges,num__rb__monthly_charges,num__rb__total_charges,num__pol__1,num__pol__monthly_charges,num__pol__total_charges,num__pol__monthly_charges^2,num__pol__monthly_charges total_charges,num__pol__total_charges^2,num__pol__monthly_charges^3,num__pol__monthly_charges^2 total_charges,num__pol__monthly_charges total_charges^2,num__pol__total_charges^3,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__internet_service_None,cat__gender_Male
0,1,7590-VHVEG,2020-01-01,NaT,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,No,Yes,No,No,No,No,Female,0,Yes,No,None,0,0.014583,0.335266,0.554993,0.095040,0.000118,0.000000e+00,0.041243,0.457057,0.459607,0.042093,1.762313e-12,0.0,0.232496,0.027400,-0.746200,-0.403038,1.0,29.85,29.85,891.0225,891.0225,8.910225e+02,26597.021625,2.659702e+04,2.659702e+04,2.659702e+04,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0,0.000116,0.094742,0.554677,0.335807,0.014658,0.000000e+00,0.004345,0.230314,0.596051,0.167842,1.447607e-03,0.0,0.392198,0.580795,-0.246891,0.145000,1.0,56.95,1889.50,3243.3025,107607.0250,3.570210e+06,184706.077375,6.128220e+06,2.033235e+08,6.745912e+09,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,3,3668-QPYBK,2019-10-01,2019-12-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1,0.000301,0.114432,0.572271,0.302499,0.010496,0.000000e+00,0.038335,0.447921,0.468533,0.045211,7.533768e-09,0.0,0.354882,0.119809,-0.304007,-0.379963,1.0,53.85,108.15,2899.8225,5823.8775,1.169642e+04,156155.441625,3.136158e+05,6.298524e+05,1.264968e+06,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,4,7795-CFOCW,2016-05-01,NaT,One year,No,Bank transfer (automatic),42.30,1840.75,DSL,Yes,No,Yes,Yes,No,No,Male,0,No,No,None,0,0.003079,0.207835,0.598672,0.188228,0.002186,0.000000e+00,0.004700,0.235853,0.595016,0.163129,1.302506e-03,0.0,0.268116,0.574104,-0.516813,0.130633,1.0,42.30,1840.75,1789.2900,77863.7250,3.388361e+06,75686.967000,3.293636e+06,1.433277e+08,6.237125e+09,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,9237-HQITU,2019-09-01,2019-11-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,No,No,No,No,No,No,Female,0,No,No,No,1,0.000000,0.034835,0.436005,0.479704,0.049456,1.530859e-07,0.036787,0.442783,0.473414,0.047016,3.681970e-08,0.0,0.506397,0.141397,0.006449,-0.367144,1.0,70.70,151.65,4998.4900,10721.6550,2.299772e+04,353393.243000,7.580210e+05,1.625939e+06,3.487605e+06,2.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [20]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer") 

2026/04/22 19:16:17 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [48]:
model = CatBoostClassifier(auto_class_weights='Balanced').fit(transformed_df, df.target) 

Learning rate set to 0.023694
0:	learn: 0.6832487	total: 70.6ms	remaining: 1m 10s
1:	learn: 0.6737476	total: 78.9ms	remaining: 39.4s
2:	learn: 0.6644371	total: 86.8ms	remaining: 28.8s
3:	learn: 0.6554898	total: 94.3ms	remaining: 23.5s
4:	learn: 0.6469891	total: 102ms	remaining: 20.3s
5:	learn: 0.6389315	total: 109ms	remaining: 18.1s
6:	learn: 0.6321042	total: 117ms	remaining: 16.6s
7:	learn: 0.6250816	total: 125ms	remaining: 15.5s
8:	learn: 0.6194701	total: 132ms	remaining: 14.6s
9:	learn: 0.6131229	total: 140ms	remaining: 13.8s
10:	learn: 0.6068478	total: 147ms	remaining: 13.2s
11:	learn: 0.6006689	total: 157ms	remaining: 12.9s
12:	learn: 0.5951145	total: 164ms	remaining: 12.5s
13:	learn: 0.5899357	total: 172ms	remaining: 12.1s
14:	learn: 0.5850319	total: 180ms	remaining: 11.8s
15:	learn: 0.5806448	total: 188ms	remaining: 11.5s
16:	learn: 0.5761549	total: 203ms	remaining: 11.8s
17:	learn: 0.5716234	total: 211ms	remaining: 11.5s
18:	learn: 0.5675114	total: 219ms	remaining: 11.3s
19:	le

39:	learn: 0.5179262	total: 386ms	remaining: 9.27s
40:	learn: 0.5164601	total: 394ms	remaining: 9.22s
41:	learn: 0.5151285	total: 406ms	remaining: 9.27s
42:	learn: 0.5140075	total: 414ms	remaining: 9.22s
43:	learn: 0.5128152	total: 422ms	remaining: 9.17s
44:	learn: 0.5115224	total: 430ms	remaining: 9.12s
45:	learn: 0.5104572	total: 437ms	remaining: 9.06s
46:	learn: 0.5091794	total: 445ms	remaining: 9.01s
47:	learn: 0.5080121	total: 452ms	remaining: 8.96s
48:	learn: 0.5071648	total: 460ms	remaining: 8.92s
49:	learn: 0.5062362	total: 467ms	remaining: 8.88s
50:	learn: 0.5051494	total: 476ms	remaining: 8.86s
51:	learn: 0.5042700	total: 484ms	remaining: 8.82s
52:	learn: 0.5035622	total: 492ms	remaining: 8.79s
53:	learn: 0.5026448	total: 501ms	remaining: 8.78s
54:	learn: 0.5018533	total: 509ms	remaining: 8.74s
55:	learn: 0.5012380	total: 516ms	remaining: 8.7s
56:	learn: 0.5005796	total: 524ms	remaining: 8.66s
57:	learn: 0.5000118	total: 531ms	remaining: 8.63s
58:	learn: 0.4993282	total: 538m

In [49]:
REGISTRY_MODEL_NAME = "churn_model_vds"
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    # ваш код здесь
    model_info = mlflow.sklearn.log_model( 
            sk_model=model,
            artifact_path='models',
            registered_model_name=REGISTRY_MODEL_NAME,
		)

Registered model 'churn_model_vds' already exists. Creating a new version of this model...
2026/04/22 19:25:24 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: churn_model_vds, version 3
Created version '3' of model 'churn_model_vds'.


In [52]:
run_id

'297710ae157c47e3be2c4624736312d3'